In [2]:
import pandas as pd
import numpy as np
import json
import random
import re
from pathlib import Path

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

BASE_DIR = Path(".")
OUTPUT_DIR = Path("region_sensitivity_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

REGION_TABLE_PATH = BASE_DIR / "region_sensitivity_table.csv"
MASTER_TABLE_PATH = BASE_DIR / "master_table_station_hour_2022_2024_benchmark_labeled.csv"
BASELINE_TABLE_PATH = BASE_DIR / "baseline_table.csv"

N_PER_TASK = 600

In [3]:
region_df = pd.read_csv(REGION_TABLE_PATH)
master_df = pd.read_csv(MASTER_TABLE_PATH)

if BASELINE_TABLE_PATH.exists():
    baseline_df = pd.read_csv(BASELINE_TABLE_PATH)
else:
    baseline_df = None

def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )
    return df

region_df = clean_columns(region_df)
master_df = clean_columns(master_df)

if baseline_df is not None:
    baseline_df = clean_columns(baseline_df)

print("region_df shape:", region_df.shape)
print("master_df shape:", master_df.shape)

print("\nregion_df columns:")
print(region_df.columns.tolist())

print("\nmaster_df columns:")
print(master_df.columns.tolist())

/var/folders/yy/qt60dp0d7b11zpqr_tpfvkxm0000gn/T/ipykernel_74158/1597582912.py:2: DtypeWarning: Columns (12,29,31,64,65,66) have mixed types. Specify dtype option on import or set low_memory=False.
  master_df = pd.read_csv(MASTER_TABLE_PATH)


region_df shape: (1079, 11)
master_df shape: (1395190, 71)

region_df columns:
['lga', 'weather_condition', 'n_observations', 'n_stations', 'avg_expected_volume', 'avg_observed_volume', 'avg_temperature_c', 'avg_rain_mm', 'sensitivity_score', 'median_abs_change_pct', 'significant_change_rate']

master_df columns:
['station_key', 'date', 'hour', 'volume', 'public_holiday', 'school_holiday', 'crash_count', 'has_crash', 'crash_severity_sum', 'crash_injury_sum', 'crash_fatal_count', 'crash_wet_count', 'station_id', 'name', 'road_name', 'intersection', 'suburb', 'lga', 'post_code', 'wgs84_latitude', 'wgs84_longitude', 'poi_food_count_500m', 'poi_education_count_500m', 'poi_healthcare_count_500m', 'poi_public_transport_count_500m', 'poi_leisure_count_500m', 'poi_tourism_count_500m', 'poi_shop_count_500m', 'building_count_500m', 'landuse_top1_category_500m', 'landuse_top1_ratio_500m', 'landuse_top2_category_500m', 'landuse_top2_ratio_500m', 'poi_missing_data', 'building_missing_data', 'landus

In [5]:
# -----------------------------
# Manually define key columns based on current data
# -----------------------------

# region_sensitivity_table columns
LGA_COL = "lga"
WEATHER_COL = "weather_condition"

SENSITIVITY_COL = "sensitivity_score"

SIG_RATE_COL = "significant_change_rate"
N_OBS_COL = "n_observations"
N_STATIONS_COL = "n_stations"

REGION_EXPECTED_VOLUME_COL = "avg_expected_volume"
REGION_OBSERVED_VOLUME_COL = "avg_observed_volume"


# master table columns
MASTER_LGA_COL = "lga"
MASTER_WEATHER_COL = "weather_combined_label"

MASTER_STATION_COL = "station_id"

OBSERVED_VOLUME_COL = "volume"
EXPECTED_VOLUME_COL = "expected_volume"

LAND_USE_COL = "landuse_top1_category_500m"

EVENT_COL = "event_count_3km"
CRASH_COL = "crash_count"

HAS_SIGNIFICANT_CHANGE_COL = "has_significant_change"
TRAFFIC_CHANGE_PCT_COL = "traffic_change_pct"


print("Column mapping confirmed:")
print("LGA_COL:", LGA_COL)
print("WEATHER_COL:", WEATHER_COL)
print("SENSITIVITY_COL:", SENSITIVITY_COL)
print("SIG_RATE_COL:", SIG_RATE_COL)
print("N_OBS_COL:", N_OBS_COL)
print("N_STATIONS_COL:", N_STATIONS_COL)
print("MASTER_WEATHER_COL:", MASTER_WEATHER_COL)
print("OBSERVED_VOLUME_COL:", OBSERVED_VOLUME_COL)
print("EXPECTED_VOLUME_COL:", EXPECTED_VOLUME_COL)
print("LAND_USE_COL:", LAND_USE_COL)
print("EVENT_COL:", EVENT_COL)
print("CRASH_COL:", CRASH_COL)

Column mapping confirmed:
LGA_COL: lga
WEATHER_COL: weather_condition
SENSITIVITY_COL: sensitivity_score
SIG_RATE_COL: significant_change_rate
N_OBS_COL: n_observations
N_STATIONS_COL: n_stations
MASTER_WEATHER_COL: weather_combined_label
OBSERVED_VOLUME_COL: volume
EXPECTED_VOLUME_COL: expected_volume
LAND_USE_COL: landuse_top1_category_500m
EVENT_COL: event_count_3km
CRASH_COL: crash_count


In [6]:
# -----------------------------
# Cell 4: Reliable LGA filtering
# -----------------------------

MIN_OBSERVATIONS = 30
MIN_STATIONS = 2

reliable_df = region_df.copy()

reliable_df = reliable_df[
    (reliable_df[N_OBS_COL] >= MIN_OBSERVATIONS)
    & (reliable_df[N_STATIONS_COL] >= MIN_STATIONS)
].copy()

reliable_lgas = sorted(reliable_df[LGA_COL].dropna().unique())

print("Number of reliable LGAs:", len(reliable_lgas))
print(reliable_lgas)

print("\nReliable rows:", reliable_df.shape)

display(
    reliable_df[
        [
            LGA_COL,
            WEATHER_COL,
            N_OBS_COL,
            N_STATIONS_COL,
            REGION_EXPECTED_VOLUME_COL,
            REGION_OBSERVED_VOLUME_COL,
            SENSITIVITY_COL,
            SIG_RATE_COL
        ]
    ].head(10)
)

Number of reliable LGAs: 23
['Auburn', 'Bankstown', 'Blacktown', 'Canterbury', 'Fairfield', 'Holroyd', 'Hornsby', 'Ku-Ring-Gai', 'Lane Cove', 'Leichhardt', 'Marrickville', 'Mosman', 'Parramatta', 'Penrith', 'Pittwater', 'Rockdale', 'Ryde', 'Strathfield', 'Sutherland', 'Sydney', 'The Hills', 'Warringah', 'Willoughby']

Reliable rows: (450, 11)


,lga,weather_condition,n_observations,n_stations,avg_expected_volume,avg_observed_volume,sensitivity_score,significant_change_rate
40,Ryde,heavy_rain+very_humid+overcast,37,6,1076.270270,1101.081081,46.882432,0.864865
41,Sutherland,moderate_rain+overcast,31,5,852.387097,869.774194,46.774839,0.516129
48,Ryde,light_rain+strong_wind+very_humid+overcast,111,5,1297.621622,1389.297297,43.608829,0.738739
63,Sutherland,no_rain+hot,480,5,883.146875,1008.660417,39.430729,0.618750
66,Sutherland,no_rain+hot+overcast,154,5,876.061688,916.220779,38.475130,0.558442
72,Strathfield,no_rain+hot,190,3,1643.294737,1864.947368,37.472368,0.489474
86,Ryde,light_rain+strong_wind,162,6,1633.410494,2080.592593,35.290494,0.450617
89,Ryde,no_rain+strong_wind+very_humid,32,4,1006.328125,1090.968750,34.510312,0.593750
92,Parramatta,no_rain+strong_wind,842,6,2407.697743,2439.564133,34.330582,0.477435
93,Ryde,light_rain+very_humid,880,6,1039.062500,1205.551136,34.309273,0.603409


In [7]:
# -----------------------------
# Cell 5: Check weather label matching
# -----------------------------

region_weather_values = set(
    reliable_df[WEATHER_COL]
    .dropna()
    .astype(str)
    .unique()
)

master_weather_values = set(
    master_df[MASTER_WEATHER_COL]
    .dropna()
    .astype(str)
    .unique()
)

overlap_weather_values = region_weather_values.intersection(master_weather_values)

print("Number of weather labels in reliable_df:", len(region_weather_values))
print("Number of weather labels in master_df:", len(master_weather_values))
print("Number of overlapping weather labels:", len(overlap_weather_values))

print("\nSample region weather labels:")
print(sorted(list(region_weather_values))[:20])

print("\nSample master weather labels:")
print(sorted(list(master_weather_values))[:20])

print("\nSample overlapping weather labels:")
print(sorted(list(overlap_weather_values))[:20])

Number of weather labels in reliable_df: 31
Number of weather labels in master_df: 47
Number of overlapping weather labels: 31

Sample region weather labels:
['heavy_rain+very_humid+overcast', 'light_rain', 'light_rain+hot', 'light_rain+hot+overcast', 'light_rain+overcast', 'light_rain+strong_wind', 'light_rain+strong_wind+overcast', 'light_rain+strong_wind+very_humid', 'light_rain+strong_wind+very_humid+overcast', 'light_rain+very_humid', 'light_rain+very_humid+overcast', 'moderate_rain', 'moderate_rain+overcast', 'moderate_rain+strong_wind+very_humid+overcast', 'moderate_rain+very_humid', 'moderate_rain+very_humid+overcast', 'no_rain', 'no_rain+extreme_heat', 'no_rain+extreme_heat+overcast', 'no_rain+extreme_heat+strong_wind']

Sample master weather labels:
['heavy_rain', 'heavy_rain+overcast', 'heavy_rain+strong_wind+very_humid', 'heavy_rain+strong_wind+very_humid+overcast', 'heavy_rain+very_humid', 'heavy_rain+very_humid+overcast', 'light_rain', 'light_rain+hot', 'light_rain+hot+ov

In [8]:
# -----------------------------
# Cell 6: Aggregate context features for QA prompts
# -----------------------------

def safe_mode(series):
    series = series.dropna().astype(str)
    if len(series) == 0:
        return "not available"
    return series.mode().iloc[0]


def label_by_quantile(value, q1, q2):
    if pd.isna(value):
        return "not available"
    if value <= q1:
        return "low"
    elif value <= q2:
        return "moderate"
    else:
        return "high"


# ============================================================
# 1. Traffic / event / crash context by LGA + weather condition
# ============================================================

traffic_context = (
    master_df
    .groupby([MASTER_LGA_COL, MASTER_WEATHER_COL])
    .agg(
        avg_volume_from_master=(OBSERVED_VOLUME_COL, "mean"),
        avg_expected_volume_from_master=(EXPECTED_VOLUME_COL, "mean"),
        avg_event_count_3km=(EVENT_COL, "mean"),
        avg_crash_count=(CRASH_COL, "mean"),
        significant_change_rate_from_master=(HAS_SIGNIFICANT_CHANGE_COL, "mean"),
        avg_traffic_change_pct_from_master=(TRAFFIC_CHANGE_PCT_COL, "mean"),
        n_stations_from_master=(MASTER_STATION_COL, pd.Series.nunique)
    )
    .reset_index()
    .rename(columns={
        MASTER_LGA_COL: LGA_COL,
        MASTER_WEATHER_COL: WEATHER_COL
    })
)

print("traffic_context shape:", traffic_context.shape)

# ============================================================
# 2. Static LGA context: land use + POI density
# Use one row per station to avoid repeated hourly rows dominating POI counts.
# ============================================================

poi_cols = [
    c for c in master_df.columns
    if c.startswith("poi_") and c.endswith("_count_500m")
]

print("\nDetected POI columns:")
print(poi_cols)

station_static_cols = [
    "station_key",
    MASTER_STATION_COL,
    MASTER_LGA_COL,
    LAND_USE_COL
] + poi_cols

station_static_cols = [c for c in station_static_cols if c in master_df.columns]

station_static = (
    master_df[station_static_cols]
    .drop_duplicates()
    .copy()
)

if len(poi_cols) > 0:
    station_static["total_poi_count_500m"] = station_static[poi_cols].sum(axis=1)
else:
    station_static["total_poi_count_500m"] = np.nan


lga_context = (
    station_static
    .groupby(MASTER_LGA_COL)
    .agg(
        dominant_land_use=(LAND_USE_COL, safe_mode),
        avg_total_poi_count_500m=("total_poi_count_500m", "mean"),
        n_static_stations=(MASTER_STATION_COL, pd.Series.nunique)
    )
    .reset_index()
    .rename(columns={MASTER_LGA_COL: LGA_COL})
)

# POI density label
q1 = lga_context["avg_total_poi_count_500m"].quantile(0.33)
q2 = lga_context["avg_total_poi_count_500m"].quantile(0.66)

lga_context["poi_density"] = lga_context["avg_total_poi_count_500m"].apply(
    lambda x: label_by_quantile(x, q1, q2)
)

print("\nlga_context shape:", lga_context.shape)

display(traffic_context.head())
display(lga_context.head())

traffic_context shape: (1081, 9)

Detected POI columns:
['poi_food_count_500m', 'poi_education_count_500m', 'poi_healthcare_count_500m', 'poi_public_transport_count_500m', 'poi_leisure_count_500m', 'poi_tourism_count_500m', 'poi_shop_count_500m']

lga_context shape: (32, 5)


,lga,weather_condition,avg_volume_from_master,avg_expected_volume_from_master,avg_event_count_3km,avg_crash_count,significant_change_rate_from_master,avg_traffic_change_pct_from_master,n_stations_from_master
0,Auburn,heavy_rain+overcast,1259.500000,1280.750000,0.0,2.500000,0.000000,-1.700000,2
1,Auburn,heavy_rain+strong_wind+very_humid+overcast,83.000000,190.250000,0.0,0.000000,0.750000,-52.530000,2
2,Auburn,heavy_rain+very_humid,658.750000,677.750000,0.0,1.250000,0.250000,-11.580000,2
3,Auburn,heavy_rain+very_humid+overcast,934.000000,1129.821429,0.0,3.000000,0.357143,-12.705714,3
4,Auburn,light_rain,1215.034991,1280.025783,0.0,2.865562,0.263352,-3.788766,4


,lga,dominant_land_use,avg_total_poi_count_500m,n_static_stations,poi_density
0,Auburn,residential,37.750000,4,moderate
1,Bankstown,residential,17.857143,7,low
2,Blacktown,residential,8.000000,2,low
3,Burwood,residential,51.000000,1,moderate
4,Campbelltown,residential,0.000000,1,low


In [9]:
# -----------------------------
# Cell 7: Merge context back to reliable region table
# -----------------------------

qa_base = reliable_df.copy()

qa_base = qa_base.merge(
    traffic_context,
    on=[LGA_COL, WEATHER_COL],
    how="left"
)

qa_base = qa_base.merge(
    lga_context,
    on=LGA_COL,
    how="left"
)


# ============================================================
# Display fields for prompt
# These are allowed to be shown to the model.
# They are contextual evidence, not the internal gold-answer score.
# ============================================================

qa_base["display_n_stations"] = qa_base[N_STATIONS_COL]

qa_base["display_avg_expected_volume"] = qa_base[REGION_EXPECTED_VOLUME_COL]
qa_base["display_avg_observed_volume"] = qa_base[REGION_OBSERVED_VOLUME_COL]

qa_base["display_dominant_land_use"] = qa_base["dominant_land_use"].fillna("not available")
qa_base["display_poi_density"] = qa_base["poi_density"].fillna("not available")


# Significant change frequency shown as low / moderate / high, not raw score
q1 = qa_base[SIG_RATE_COL].quantile(0.33)
q2 = qa_base[SIG_RATE_COL].quantile(0.66)

qa_base["display_significant_change_frequency"] = qa_base[SIG_RATE_COL].apply(
    lambda x: label_by_quantile(x, q1, q2)
)

# Typical traffic deviation magnitude shown as low / moderate / high
# This is based on median_abs_change_pct, not sensitivity_score.
DEVIATION_MAG_COL = "median_abs_change_pct"

q1_dev = qa_base[DEVIATION_MAG_COL].quantile(0.33)
q2_dev = qa_base[DEVIATION_MAG_COL].quantile(0.66)

qa_base["display_traffic_deviation_magnitude"] = qa_base[DEVIATION_MAG_COL].apply(
    lambda x: label_by_quantile(x, q1_dev, q2_dev)
)

# Event and crash context
qa_base["display_avg_event_count_3km"] = qa_base["avg_event_count_3km"]
qa_base["display_avg_crash_count"] = qa_base["avg_crash_count"]


# Traffic change direction label
def describe_observed_vs_expected(row):
    expected = row["display_avg_expected_volume"]
    observed = row["display_avg_observed_volume"]

    if pd.isna(expected) or pd.isna(observed):
        return "not available"

    diff_pct = (observed - expected) / expected * 100 if expected != 0 else np.nan

    if pd.isna(diff_pct):
        return "not available"
    elif diff_pct >= 10:
        return "much higher than typical"
    elif diff_pct >= 3:
        return "higher than typical"
    elif diff_pct <= -10:
        return "much lower than typical"
    elif diff_pct <= -3:
        return "lower than typical"
    else:
        return "close to typical"


qa_base["display_observed_vs_expected"] = qa_base.apply(
    describe_observed_vs_expected,
    axis=1
)


# ============================================================
# Check missing values in display fields
# ============================================================

display_cols = [
    LGA_COL,
    WEATHER_COL,
    "display_n_stations",
    "display_avg_expected_volume",
    "display_avg_observed_volume",
    "display_observed_vs_expected",
    "display_dominant_land_use",
    "display_poi_density",
    "display_significant_change_frequency",
    "display_traffic_deviation_magnitude",
    "display_avg_event_count_3km",
    "display_avg_crash_count",
    SENSITIVITY_COL
]

print("qa_base shape:", qa_base.shape)

print("\nMissing values in display columns:")
print(qa_base[display_cols].isna().sum())

display(qa_base[display_cols].head(10))

qa_base shape: (450, 32)

Missing values in display columns:
lga                                     0
weather_condition                       0
display_n_stations                      0
display_avg_expected_volume             0
display_avg_observed_volume             0
display_observed_vs_expected            0
display_dominant_land_use               0
display_poi_density                     0
display_significant_change_frequency    0
display_traffic_deviation_magnitude     0
display_avg_event_count_3km             0
display_avg_crash_count                 0
sensitivity_score                       0
dtype: int64


,lga,weather_condition,display_n_stations,display_avg_expected_volume,display_avg_observed_volume,display_observed_vs_expected,display_dominant_land_use,display_poi_density,display_significant_change_frequency,display_traffic_deviation_magnitude,display_avg_event_count_3km,display_avg_crash_count,sensitivity_score
0,Ryde,heavy_rain+very_humid+overcast,6,1076.270270,1101.081081,close to typical,residential,high,high,high,0.000000,1.270270,46.882432
1,Sutherland,moderate_rain+overcast,5,852.387097,869.774194,close to typical,residential,low,high,high,0.000000,0.677419,46.774839
2,Ryde,light_rain+strong_wind+very_humid+overcast,5,1297.621622,1389.297297,higher than typical,residential,high,high,high,0.000000,1.324324,43.608829
3,Sutherland,no_rain+hot,5,883.146875,1008.660417,much higher than typical,residential,low,high,high,0.000000,1.206250,39.430729
4,Sutherland,no_rain+hot+overcast,5,876.061688,916.220779,higher than typical,residential,low,high,high,0.000000,1.142857,38.475130
5,Strathfield,no_rain+hot,3,1643.294737,1864.947368,much higher than typical,industrial,high,moderate,moderate,0.000000,2.478947,37.472368
6,Ryde,light_rain+strong_wind,6,1633.410494,2080.592593,much higher than typical,residential,high,moderate,moderate,0.000000,1.938272,35.290494
7,Ryde,no_rain+strong_wind+very_humid,4,1006.328125,1090.968750,higher than typical,residential,high,high,high,0.000000,0.906250,34.510312
8,Parramatta,no_rain+strong_wind,6,2407.697743,2439.564133,close to typical,residential,moderate,moderate,moderate,0.007084,2.870130,34.330582
9,Ryde,light_rain+very_humid,6,1039.062500,1205.551136,much higher than typical,residential,high,high,high,0.000000,1.088964,34.309273


In [10]:
# -----------------------------
# Cell 8: Prompt formatting functions
# -----------------------------

OPTION_LETTERS = ["A", "B", "C", "D"]

SYSTEM_PROMPT = (
    "You are evaluating weather-sensitive urban traffic patterns using the provided "
    "regional context and traffic evidence. Use only the information provided in the question. "
    "Do not rely on external assumptions about the LGA names. "
    "A weather-sensitive traffic region is one whose observed traffic appears to deviate "
    "more strongly or more frequently from its expected traffic level under the given weather condition. "
    "Return only one option letter from the given options."
)

def fmt_num(x, digits=1):
    if pd.isna(x):
        return "not available"
    try:
        return f"{float(x):.{digits}f}"
    except Exception:
        return str(x)


def format_candidate_context(letter, row):
    lga = row[LGA_COL]

    n_stations = fmt_num(row["display_n_stations"], digits=0)
    avg_expected = fmt_num(row["display_avg_expected_volume"], digits=1)
    avg_observed = fmt_num(row["display_avg_observed_volume"], digits=1)

    observed_vs_expected = row["display_observed_vs_expected"]
    land_use = row["display_dominant_land_use"]
    poi_density = row["display_poi_density"]
    change_freq = row["display_significant_change_frequency"]
    deviation_mag = row["display_traffic_deviation_magnitude"]

    event_count = fmt_num(row["display_avg_event_count_3km"], digits=1)
    crash_count = fmt_num(row["display_avg_crash_count"], digits=1)

    lines = [
        f"{letter}. {lga}",
        f"- Number of traffic stations: {n_stations}",
        f"- Average expected traffic volume: {avg_expected}",
        f"- Average observed traffic volume under this weather condition: {avg_observed}",
        f"- Observed traffic compared with typical level: {observed_vs_expected}",
        f"- Dominant land-use type: {land_use}",
        f"- POI density: {poi_density}",
        f"- Significant traffic change frequency: {change_freq}",
        f"- Typical traffic deviation magnitude: {deviation_mag}",
        f"- Average nearby event count within 3 km: {event_count}",
        f"- Average nearby crash count: {crash_count}",
    ]

    return "\n".join(lines)


def build_options_text(candidate_rows):
    option_lines = []

    for letter, (_, row) in zip(OPTION_LETTERS, candidate_rows.iterrows()):
        option_lines.append(f"{letter}. {row[LGA_COL]}")

    return "\n".join(option_lines)


def build_question(task_type, weather_condition, candidate_rows):
    candidate_blocks = []

    for letter, (_, row) in zip(OPTION_LETTERS, candidate_rows.iterrows()):
        candidate_blocks.append(format_candidate_context(letter, row))

    candidate_text = "\n\n".join(candidate_blocks)
    options_text = build_options_text(candidate_rows)

    if task_type == "pairwise_comparison":
        question_text = (
            f"Weather condition:\n"
            f"{weather_condition}\n\n"
            f"Candidate LGAs:\n\n"
            f"{candidate_text}\n\n"
            f"Question:\n"
            f"Based on the regional context and traffic evidence above, "
            f"which LGA appears to have more weather-sensitive traffic under {weather_condition} conditions?\n\n"
            f"Options:\n"
            f"{options_text}\n\n"
            f"Return only one option letter."
        )

    elif task_type == "top_sensitive_region":
        question_text = (
            f"Weather condition:\n"
            f"{weather_condition}\n\n"
            f"Candidate LGAs:\n\n"
            f"{candidate_text}\n\n"
            f"Question:\n"
            f"Based on the regional context and traffic evidence above, "
            f"which LGA appears to have the most weather-sensitive traffic under {weather_condition} conditions?\n\n"
            f"Options:\n"
            f"{options_text}\n\n"
            f"Return only one option letter."
        )

    elif task_type == "management_priority":
        question_text = (
            f"Weather condition:\n"
            f"{weather_condition}\n\n"
            f"Candidate LGAs:\n\n"
            f"{candidate_text}\n\n"
            f"Question:\n"
            f"Based on the regional context and traffic evidence above, "
            f"which LGA should be prioritised for weather-sensitive traffic management under "
f"{weather_condition} conditions, because its traffic appears more affected by the given weather?\n\n"
            f"Options:\n"
            f"{options_text}\n\n"
            f"Return only one option letter."
        )

    else:
        raise ValueError(f"Unknown task_type: {task_type}")

    return question_text


print("System prompt:")
print(SYSTEM_PROMPT)

print("\nFormatting functions are ready.")

System prompt:
You are evaluating weather-sensitive urban traffic patterns using the provided regional context and traffic evidence. Use only the information provided in the question. Do not rely on external assumptions about the LGA names. A weather-sensitive traffic region is one whose observed traffic appears to deviate more strongly or more frequently from its expected traffic level under the given weather condition. Return only one option letter from the given options.

Formatting functions are ready.


In [11]:
# -----------------------------
# Cell 9: Internal metrics and QA generation functions
# -----------------------------

def minmax_scale(series):
    series = series.astype(float)
    min_val = series.min()
    max_val = series.max()

    if max_val == min_val:
        return pd.Series(0.5, index=series.index)

    return (series - min_val) / (max_val - min_val)


# ============================================================
# Internal priority score
# This is NOT shown to the model.
# It is only used to decide the gold answer for management_priority.
# ============================================================

qa_base["internal_priority_score"] = (
    0.55 * minmax_scale(qa_base[SENSITIVITY_COL])
    + 0.25 * minmax_scale(qa_base[SIG_RATE_COL])
    + 0.15 * minmax_scale(qa_base["median_abs_change_pct"])
    + 0.05 * minmax_scale(qa_base["display_n_stations"])
)

INTERNAL_PRIORITY_COL = "internal_priority_score"


def get_gold_letter(candidate_rows, metric_col):
    values = candidate_rows[metric_col].astype(float).values
    gold_idx = int(np.argmax(values))
    return OPTION_LETTERS[gold_idx], candidate_rows.iloc[gold_idx][LGA_COL]


def sample_candidates_for_weather(sub_df, n_candidates, metric_col):
    """
    Sample candidate LGAs under the same weather condition.
    The gold answer is determined internally by metric_col.
    Candidate order is shuffled, so the correct letter is not fixed.
    """
    if len(sub_df) < n_candidates:
        return None

    sub_sorted = sub_df.sort_values(metric_col, ascending=False).copy()

    if n_candidates == 2:
        top_pool = sub_sorted.head(max(2, len(sub_sorted) // 2))
        lower_pool = sub_sorted.tail(max(2, len(sub_sorted) // 2))

        row_1 = top_pool.sample(1, random_state=random.randint(0, 10**6))
        row_2 = lower_pool.sample(1, random_state=random.randint(0, 10**6))

        candidates = pd.concat([row_1, row_2]).drop_duplicates(subset=[LGA_COL])

        if len(candidates) < 2:
            candidates = sub_df.sample(
                2,
                random_state=random.randint(0, 10**6),
                replace=False
            )

    else:
        top_pool = sub_sorted.head(max(2, len(sub_sorted) // 3))
        lower_pool = sub_sorted.iloc[max(1, len(sub_sorted) // 3):]

        gold_candidate = top_pool.sample(
            1,
            random_state=random.randint(0, 10**6)
        )

        if len(lower_pool) >= n_candidates - 1:
            distractors = lower_pool.sample(
                n_candidates - 1,
                random_state=random.randint(0, 10**6),
                replace=False
            )
        else:
            distractors = sub_df.drop(gold_candidate.index).sample(
                n_candidates - 1,
                random_state=random.randint(0, 10**6),
                replace=True
            )

        candidates = pd.concat([gold_candidate, distractors]).drop_duplicates(subset=[LGA_COL])

        if len(candidates) < n_candidates:
            candidates = sub_df.sample(
                n_candidates,
                random_state=random.randint(0, 10**6),
                replace=False
            )

    candidates = candidates.sample(
        frac=1,
        random_state=random.randint(0, 10**6)
    ).reset_index(drop=True)

    return candidates


def generate_task_qa(task_type, n_samples):
    qa_items = []

    weather_conditions = sorted(qa_base[WEATHER_COL].dropna().unique())

    if task_type == "pairwise_comparison":
        n_candidates = 2
        metric_col = SENSITIVITY_COL

    elif task_type == "top_sensitive_region":
        n_candidates = 4
        metric_col = SENSITIVITY_COL

    elif task_type == "management_priority":
        n_candidates = 4
        metric_col = INTERNAL_PRIORITY_COL

    else:
        raise ValueError(f"Unknown task_type: {task_type}")

    attempts = 0
    max_attempts = n_samples * 100

    while len(qa_items) < n_samples and attempts < max_attempts:
        attempts += 1

        weather = random.choice(weather_conditions)
        sub_df = qa_base[qa_base[WEATHER_COL] == weather].copy()
        sub_df = sub_df.dropna(subset=[metric_col])

        if len(sub_df) < n_candidates:
            continue

        candidates = sample_candidates_for_weather(
            sub_df=sub_df,
            n_candidates=n_candidates,
            metric_col=metric_col
        )

        if candidates is None:
            continue

        gold_letter, gold_region = get_gold_letter(candidates, metric_col)

        question = build_question(
            task_type=task_type,
            weather_condition=weather,
            candidate_rows=candidates
        )

        options = {
            letter: row[LGA_COL]
            for letter, (_, row) in zip(OPTION_LETTERS, candidates.iterrows())
        }

        qa_id = f"rs_context_v2_{task_type}_{len(qa_items) + 1:04d}"

        qa_items.append({
            "id": qa_id,
            "task_type": task_type,
            "system_prompt": SYSTEM_PROMPT,
            "question": question,
            "options": options,
            "answer": gold_letter,
            "gold_region": gold_region,
            "weather_condition": weather,
            "candidate_lgas": list(options.values())
        })

    if len(qa_items) < n_samples:
        print(f"Warning: only generated {len(qa_items)} / {n_samples} for {task_type}")

    return qa_items


print("Internal priority score created.")
print("Priority score range:")
print(qa_base[INTERNAL_PRIORITY_COL].describe())

print("\nQA generation functions are ready.")

Internal priority score created.
Priority score range:
count    450.000000
mean       0.357801
std        0.143669
min        0.004336
25%        0.256111
50%        0.351972
75%        0.441628
max        0.952448
Name: internal_priority_score, dtype: float64

QA generation functions are ready.


In [12]:
# -----------------------------
# Cell 10: Generate 1800 context-rich QA examples
# -----------------------------

N_PER_TASK = 600

pairwise_qa = generate_task_qa("pairwise_comparison", N_PER_TASK)
top_region_qa = generate_task_qa("top_sensitive_region", N_PER_TASK)
priority_qa = generate_task_qa("management_priority", N_PER_TASK)

all_qa = pairwise_qa + top_region_qa + priority_qa

qa_df = pd.DataFrame(all_qa)

print("Total QA:", len(qa_df))

print("\nTask type distribution:")
print(qa_df["task_type"].value_counts())

print("\nAnswer distribution by task:")
print(pd.crosstab(qa_df["task_type"], qa_df["answer"]))

print("\nWeather condition coverage:")
print(qa_df["weather_condition"].nunique(), "weather conditions")

display(qa_df.head())

Total QA: 1800

Task type distribution:
task_type
pairwise_comparison     600
top_sensitive_region    600
management_priority     600
Name: count, dtype: int64

Answer distribution by task:
answer                  A    B    C    D
task_type                               
management_priority   147  163  131  159
pairwise_comparison   303  297    0    0
top_sensitive_region  153  159  134  154

Weather condition coverage:
27 weather conditions


,id,task_type,system_prompt,question,options,answer,gold_region,weather_condition,candidate_lgas
0,rs_context_v2_pairwise_comparison_0001,pairwise_comparison,You are evaluating weather-sensitive urban tra...,Weather condition:\nno_rain+hot\n\nCandidate L...,"{'A': 'Ku-Ring-Gai', 'B': 'Mosman'}",A,Ku-Ring-Gai,no_rain+hot,"[Ku-Ring-Gai, Mosman]"
1,rs_context_v2_pairwise_comparison_0002,pairwise_comparison,You are evaluating weather-sensitive urban tra...,Weather condition:\nlight_rain+strong_wind+ver...,"{'A': 'Sydney', 'B': 'Hornsby'}",B,Hornsby,light_rain+strong_wind+very_humid+overcast,"[Sydney, Hornsby]"
2,rs_context_v2_pairwise_comparison_0003,pairwise_comparison,You are evaluating weather-sensitive urban tra...,Weather condition:\nno_rain+hot+overcast\n\nCa...,"{'A': 'Rockdale', 'B': 'Penrith'}",B,Penrith,no_rain+hot+overcast,"[Rockdale, Penrith]"
3,rs_context_v2_pairwise_comparison_0004,pairwise_comparison,You are evaluating weather-sensitive urban tra...,Weather condition:\nno_rain+extreme_heat+overc...,"{'A': 'Holroyd', 'B': 'Hornsby'}",A,Holroyd,no_rain+extreme_heat+overcast,"[Holroyd, Hornsby]"
4,rs_context_v2_pairwise_comparison_0005,pairwise_comparison,You are evaluating weather-sensitive urban tra...,Weather condition:\nlight_rain+strong_wind+ove...,"{'A': 'Leichhardt', 'B': 'Willoughby'}",A,Leichhardt,light_rain+strong_wind+overcast,"[Leichhardt, Willoughby]"


In [13]:
# -----------------------------
# Cell 11: Check prompt leakage
# -----------------------------

FORBIDDEN_TERMS = [
    "sensitivity_score",
    "priority_score",
    "internal_priority_score",
    "gold_answer",
    "correct_option",
    "gold_region",
    "answer:",
    "gold region",
    "score =",
    "priority ="
]


def check_prompt_leakage(df):
    bad_rows = []

    for _, row in df.iterrows():
        question_text = str(row["question"]).lower()
        system_text = str(row["system_prompt"]).lower()

        for term in FORBIDDEN_TERMS:
            if term.lower() in question_text:
                bad_rows.append((row["id"], "question", term))

            if term.lower() in system_text:
                bad_rows.append((row["id"], "system_prompt", term))

    return bad_rows


bad_rows = check_prompt_leakage(qa_df)

if len(bad_rows) == 0:
    print("No leakage found.")
else:
    print("Potential leakage found:")
    print(bad_rows[:30])
    print("Total leakage cases:", len(bad_rows))

No leakage found.


In [14]:
# -----------------------------
# Cell 12: Inspect sample questions from each task type
# -----------------------------

for task_type in ["pairwise_comparison", "top_sensitive_region", "management_priority"]:
    print("=" * 100)
    print("TASK:", task_type)
    print("=" * 100)

    sample_row = qa_df[qa_df["task_type"] == task_type].sample(
        1,
        random_state=RANDOM_SEED
    ).iloc[0]

    print(sample_row["question"])
    print("\nStored answer for evaluation:", sample_row["answer"])
    print("Stored gold region:", sample_row["gold_region"])
    print()

TASK: pairwise_comparison
Weather condition:
light_rain+strong_wind+overcast

Candidate LGAs:

A. Hornsby
- Number of traffic stations: 9
- Average expected traffic volume: 1096.4
- Average observed traffic volume under this weather condition: 895.1
- Observed traffic compared with typical level: much lower than typical
- Dominant land-use type: residential
- POI density: moderate
- Significant traffic change frequency: high
- Typical traffic deviation magnitude: high
- Average nearby event count within 3 km: 0.0
- Average nearby crash count: 0.7

B. Mosman
- Number of traffic stations: 2
- Average expected traffic volume: 2276.6
- Average observed traffic volume under this weather condition: 2176.9
- Observed traffic compared with typical level: lower than typical
- Dominant land-use type: residential
- POI density: high
- Significant traffic change frequency: low
- Typical traffic deviation magnitude: low
- Average nearby event count within 3 km: 0.0
- Average nearby crash count: 1.0

In [33]:
# -----------------------------
# Cell 13: Save full 1800 context_v2 QA as official eval set
# -----------------------------

from pathlib import Path
import json

OUTPUT_DIR = Path("region_sensitivity_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FULL_JSON_PATH = OUTPUT_DIR / "region_sensitivity_qa_pairs_context_v2_1800_eval.json"
FULL_CSV_PATH = OUTPUT_DIR / "region_sensitivity_qa_preview_context_v2_1800_eval.csv"

print("Total QA:", len(qa_df))
print(qa_df["task_type"].value_counts())

# Make sure each task has 600 QA
expected_counts = {
    "pairwise_comparison": 600,
    "top_sensitive_region": 600,
    "management_priority": 600
}

actual_counts = qa_df["task_type"].value_counts().to_dict()

assert len(qa_df) == 1800, f"Expected 1800 QA pairs, but got {len(qa_df)}"

for task_type, expected_count in expected_counts.items():
    actual_count = actual_counts.get(task_type, 0)
    assert actual_count == expected_count, (
        f"{task_type}: expected {expected_count}, got {actual_count}"
    )

# Save JSON for model evaluation
json_records = json.loads(
    qa_df.to_json(orient="records", force_ascii=False)
)

with open(FULL_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(json_records, f, ensure_ascii=False, indent=2)

# Save CSV preview for manual inspection
preview_cols = [
    col for col in [
        "id",
        "task_type",
        "weather_condition",
        "question",
        "answer",
        "gold_region"
    ]
    if col in qa_df.columns
]

qa_df[preview_cols].to_csv(FULL_CSV_PATH, index=False)

print("Saved official 1800 eval JSON to:", FULL_JSON_PATH)
print("Saved official 1800 eval preview CSV to:", FULL_CSV_PATH)

Total QA: 1800
task_type
pairwise_comparison     600
top_sensitive_region    600
management_priority     600
Name: count, dtype: int64
Saved official 1800 eval JSON to: region_sensitivity_outputs/region_sensitivity_qa_pairs_context_v2_1800_eval.json
Saved official 1800 eval preview CSV to: region_sensitivity_outputs/region_sensitivity_qa_preview_context_v2_1800_eval.csv


In [1]:
# Check ground-truth answer distribution for Task 5

import pandas as pd

# First, try to use the QA DataFrame that has already been generated in the notebook
possible_names = [
    "qa_df",
    "qa_pairs_df",
    "benchmark_df",
    "eval_df",
    "df_qa",
    "qa_pairs"
]

qa_data = None
used_name = None

for name in possible_names:
    if name in globals():
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            qa_data = obj.copy()
            used_name = name
            break

        if isinstance(obj, list):
            qa_data = pd.DataFrame(obj)
            used_name = name
            break

if qa_data is None:
    # If it cannot be found in memory, read the final evaluation JSON directly.
    json_path = (
        "region_sensitivity_outputs/"
        "region_sensitivity_qa_pairs_context_v2_1800_eval.json"
    )

    qa_data = pd.read_json(json_path)
    used_name = json_path

print("Data source used:", used_name)
print("Shape:", qa_data.shape)
print("Columns:", qa_data.columns.tolist())

required_columns = ["task_type", "answer"]

# Some files may use the column name gold_answer
if "answer" not in qa_data.columns and "gold_answer" in qa_data.columns:
    qa_data = qa_data.rename(columns={"gold_answer": "answer"})

missing = [col for col in required_columns if col not in qa_data.columns]

if missing:
    raise KeyError(
        f"Missing columns: {missing}\n"
        f"Available columns: {qa_data.columns.tolist()}"
    )

qa_data["task_type"] = (
    qa_data["task_type"]
    .astype(str)
    .str.strip()
)

qa_data["answer"] = (
    qa_data["answer"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print("\n" + "=" * 70)
print("GROUND-TRUTH COUNTS BY TASK")
print("=" * 70)

count_table = pd.crosstab(
    qa_data["task_type"],
    qa_data["answer"]
).reindex(
    columns=["A", "B", "C", "D"],
    fill_value=0
)

display(count_table)

print("\n" + "=" * 70)
print("GROUND-TRUTH PERCENTAGES BY TASK")
print("=" * 70)

percentage_table = (
    pd.crosstab(
        qa_data["task_type"],
        qa_data["answer"],
        normalize="index"
    )
    .reindex(
        columns=["A", "B", "C", "D"],
        fill_value=0
    )
    * 100
).round(2)

display(percentage_table)

print("\n" + "=" * 70)
print("TASK TOTALS")
print("=" * 70)

display(
    qa_data["task_type"]
    .value_counts()
    .rename_axis("Task Type")
    .reset_index(name="Total Instances")
)

Data source used: region_sensitivity_outputs/region_sensitivity_qa_pairs_context_v2_1800_eval.json
Shape: (1800, 9)
Columns: ['id', 'task_type', 'system_prompt', 'question', 'options', 'answer', 'gold_region', 'weather_condition', 'candidate_lgas']

GROUND-TRUTH COUNTS BY TASK


answer,A,B,C,D
task_type,,,,
management_priority,162,157,150,131
pairwise_comparison,293,307,0,0
top_sensitive_region,161,144,142,153



GROUND-TRUTH PERCENTAGES BY TASK


answer,A,B,C,D
task_type,,,,
management_priority,27.00,26.17,25.00,21.83
pairwise_comparison,48.83,51.17,0.00,0.00
top_sensitive_region,26.83,24.00,23.67,25.50



TASK TOTALS


,Task Type,Total Instances
0,pairwise_comparison,600
1,top_sensitive_region,600
2,management_priority,600
